In [ ]:
import torch

In [ ]:
# Реализуйте scaled dot-product attention.
import math
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    key_padding_mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    # Вычислите QK^T и разделите оценки на sqrt(d_k).
    d_k = query.size(-1)

    # Транспонируйте K с помощью key.transpose(-2, -1).
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    # Маскируем padding-позиции.
    if key_padding_mask is not None:
        mask = key_padding_mask

        while mask.dim() < scores.dim():
            mask = mask.unsqueeze(1)

        scores = scores.masked_fill(mask, float("-inf"))

    # Примените softmax по последнему измерению.
    attention_weights = F.softmax(scores, dim=-1)

    # Вычисляем взвешенную сумму Value.
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

In [6]:
# Реализуйте multi-head self-attention с объединённой проекцией QKV.
from typing import TypeAlias

TensorPair: TypeAlias = tuple[torch.Tensor, torch.Tensor]


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embedding_dim: int, num_heads: int) -> None:
        super().__init__()
        assert embedding_dim % num_heads == 0

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.qkv = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.out_proj = nn.Linear(embedding_dim, embedding_dim)

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
    ) -> TensorPair:
        batch_size, seq_len, _ = x.shape

        # Одним Linear вычисляем Q, K и V.
        qkv = self.qkv(x)

        # [B, T, 3*E] -> [B, T, 3, H, D].
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)

        # [B, T, 3, H, D] -> [3, B, H, T, D].
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Разделяем Q, K, V.
        query, key, value = qkv.unbind(0)

        # Примените scaled_dot_product_attention к каждой голове.
        attention_output, attention_weights = scaled_dot_product_attention(
            query,
            key,
            value,
            key_padding_mask,
        )

        # [B, H, T, D] -> [B, T, H, D] -> [B, T, E].
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.reshape(
            batch_size, seq_len, self.embedding_dim
        )

        # Финальная проекция.
        output = self.out_proj(attention_output)

        return output, attention_weights